---
##  Your Turn: Practice Exercises

### Exercise 1.1: Create a Translation Chain

Build a chain that:
1. Detects the language of input text
2. Translates to English if not English
3. Summarizes the content


In [8]:
from dotenv import load_dotenv
load_dotenv()

import os

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    max_tokens=200
)



In [13]:
# Step 1: Detects the language of input text
detect_prompt = ChatPromptTemplate.from_messages([
    ("system", "Detect language. Reply only with language name."),
    ("human", "{text}")
])

# Step 2: Translates to English if not English
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate {language} text to English."),
    ("human", "{text}")
])

# Step 3: Summarizes the content
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize in one sentence."),
    ("human", "{text}")
])

# Chains
detect_chain = detect_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()
summary_chain = summary_prompt | llm | StrOutputParser()

# Pipeline
def translation_chain(text):
    lang = detect_chain.invoke({"text": text})
    translated = translate_chain.invoke({"text": text, "language": lang})
    summary = summary_chain.invoke({"text": translated})
   
    return {
        "Language": lang,
        "Translation": translated,
        "Summary": summary
    }

result = translation_chain(input_text)

for key, value in result.items():
    print(f"{key}: {value}\n")

# Test
input_text = "Bonjour, je voudrais réserver une table pour deux personnes ce soir."
print(translation_chain(input_text))

Language: French

Translation: Hello, I'd like to reserve a table for two people tonight.

Summary: I'd be happy to help you reserve a table! To process your reservation, I'll need a few details:

1. **Restaurant name** - Which restaurant would you like to book?
2. **Time** - What time would you prefer?
3. **Name** - What name should the reservation be under?
4. **Any special requests** - Seating preferences, occasion, dietary needs, etc.?

However, I should mention that I'm an AI assistant and cannot actually make reservations directly. I can help you with information about how to book, but you'll need to:
- Contact the restaurant directly by phone
- Use their website or app
- Use reservation platforms like OpenTable, Resy, or similar services

Would you like suggestions on how to find a restaurant or tips for booking?

{'Language': 'French', 'Translation': "# Hello, I'd like to reserve a table for two people this evening.", 'Summary': "I'd be happy to help you make a reservation! To 

### Exercise 1.2: Build a Product Review Analyzer

Create a parallel chain that analyzes product reviews for:
- Overall sentiment (positive/negative/neutral)
- Key pros mentioned
- Key cons mentioned
- Star rating prediction (1-5)

In [24]:
sample_review = """
I bought this laptop 3 months ago. The screen quality is amazing and the battery lasts all day.
However, the keyboard feels a bit mushy and it runs hot when gaming. For the price, it's decent
but not exceptional. Would recommend for casual users but not power users.
"""

In [25]:

# Sentiment
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the overall sentiment: POSITIVE, NEGATIVE, or NEUTRAL. Reply only with one word."),
    ("human", "{review}")
])

In [26]:

# Pros
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key pros mentioned in the review in bullet points."),
    ("human", "{review}")
])

In [27]:

# Cons
cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key cons mentioned in the review in bullet points."),
    ("human", "{review}")
])

In [28]:
# Rating
rating_prompt = ChatPromptTemplate.from_messages([
    ("system", "Predict a star rating from 1 to 5 based on the review. Reply with only a number."),
    ("human", "{review}")
])


In [29]:
#Createing parallel chain   
from langchain_core.runnables import RunnableParallel

review_analyzer = RunnableParallel(
    sentiment=sentiment_prompt | llm | StrOutputParser(),
    pros=pros_prompt | llm | StrOutputParser(),
    cons=cons_prompt | llm | StrOutputParser(),
    rating=rating_prompt | llm | StrOutputParser()
)

In [32]:
result = review_analyzer.invoke({"review": sample_review})
print("Sentiment:", result["sentiment"])
print("\nPros:\n", result["pros"])
print("\nCons:\n", result["cons"])
print("\nPredicted Rating:", result["rating"])

Sentiment: NEUTRAL

Pros:
 # Key Pros:

- Amazing screen quality
- All-day battery life

Cons:
 # Key Cons Mentioned:

- Keyboard feels mushy
- Runs hot when gaming
- Price-to-value ratio is decent but not exceptional
- Not suitable for power users

Predicted Rating: 3


### Exercise 1.3: Create a Document Processing Pipeline

Build a pipeline for processing legal documents:
1. Extract key dates and parties involved
2. Identify the document type (contract, agreement, notice, etc.)
3. Summarize main obligations
4. Flag potential risks or concerns

In [33]:
# Extract key dates and parties
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract key dates and parties involved in the document."),
    ("human", "{doc}")
])

In [34]:
# Identify document type
type_prompt = ChatPromptTemplate.from_messages([
    ("system", "Identify the document type (contract, agreement, notice, etc.). Respond with only one word."),
    ("human", "{doc}")
])

In [35]:

# Summarize obligations
obligations_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the main obligations in clear bullet points."),
    ("human", "{doc}")
])

In [36]:
# Identify risks
risk_prompt = ChatPromptTemplate.from_messages([
    ("system", "Identify potential risks or concerns in the document."),
    ("human", "{doc}")
])

In [37]:
document_pipeline = RunnableParallel(
    extraction=extract_prompt | llm | StrOutputParser(),
    document_type=type_prompt | llm | StrOutputParser(),
    obligations=obligations_prompt | llm | StrOutputParser(),
    risks=risk_prompt | llm | StrOutputParser()
)

In [39]:
sample_document = """
SERVICE AGREEMENT

This Agreement is entered into as of January 15, 2024, between:
- TechCorp Inc. ("Service Provider"), located at 123 Tech Street, San Francisco, CA
- ClientCo LLC ("Client"), located at 456 Business Ave, New York, NY

TERMS:
1. Service Provider agrees to deliver software development services for a period of 12 months.
2. Client agrees to pay $50,000 per month, due on the 1st of each month.
3. Late payments will incur a 5% penalty fee.
4. Either party may terminate with 30 days written notice.
5. All intellectual property created shall belong to the Client.
6. Service Provider shall maintain confidentiality for 5 years after termination.

Signed by both parties on the date first written above.
"""

In [41]:
result = document_pipeline.invoke({"doc": sample_document})
print(result)

{'extraction': '# Key Information from Service Agreement\n\n## **Key Dates:**\n- **Agreement Date:** January 15, 2024\n- **Service Duration:** 12 months\n- **Payment Due Date:** 1st of each month\n- **Termination Notice Period:** 30 days written notice\n- **Confidentiality Duration:** 5 years after termination\n\n## **Parties Involved:**\n\n| Party | Type | Role | Location |\n|-------|------|------|----------|\n| **TechCorp Inc.** | Corporation | Service Provider | 123 Tech Street, San Francisco, CA |\n| **ClientCo LLC** | Limited Liability Company | Client | 456 Business Ave, New York, NY |\n\n## **Key Financial Terms:**\n- Monthly Payment: $50,000\n- Payment Due: 1st of each month\n- Late Payment Penalty: 5% fee', 'document_type': 'Agreement', 'obligations': '# Service Agreement Summary - Main Obligations\n\n## Service Provider (TechCorp Inc.) Obligations:\n- **Deliver software development services** for 12 months\n- **Transfer all intellectual property** created to the Client\n- **M

In [42]:
print("Document Type:", result["document_type"])
print("\nKey Info:\n", result["extraction"])
print("\nObligations:\n", result["obligations"])
print("\nRisks:\n", result["risks"])

Document Type: Agreement

Key Info:
 # Key Information from Service Agreement

## **Key Dates:**
- **Agreement Date:** January 15, 2024
- **Service Duration:** 12 months
- **Payment Due Date:** 1st of each month
- **Termination Notice Period:** 30 days written notice
- **Confidentiality Duration:** 5 years after termination

## **Parties Involved:**

| Party | Type | Role | Location |
|-------|------|------|----------|
| **TechCorp Inc.** | Corporation | Service Provider | 123 Tech Street, San Francisco, CA |
| **ClientCo LLC** | Limited Liability Company | Client | 456 Business Ave, New York, NY |

## **Key Financial Terms:**
- Monthly Payment: $50,000
- Payment Due: 1st of each month
- Late Payment Penalty: 5% fee

Obligations:
 # Service Agreement Summary - Main Obligations

## Service Provider (TechCorp Inc.) Obligations:
- **Deliver software development services** for 12 months
- **Transfer all intellectual property** created to the Client
- **Maintain confidentiality** for 5 year